In [ ]:
# !pip install mxnet
# !pip install numpy==1.20.3
# !pip install gluonnlp pandas tqdm
# !pip install sentencepiece
# !pip install onnxruntime==1.8.0
# !pip install --use-deprecated=legacy-resolver git+https://github.com/SKTBrain/KoBERT.git@master

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import gluonnlp as nlp
import numpy as np
from tqdm import tqdm, tqdm_notebook
import pandas as pd
from sklearn.model_selection import train_test_split
import random

from kobert.utils import get_tokenizer
from kobert.pytorch_kobert import get_pytorch_kobert_model

from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bertmodel, vocab = get_pytorch_kobert_model()

In [ ]:
def fix_seed(random_seed):
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)  # if use multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(random_seed)
    random.seed(random_seed)
    
random_seed = 12
fix_seed(random_seed)

In [ ]:
# Hyperparameter
max_len = 512
batch_size = 16
warmup_ratio = 0.1
num_epochs = 8
max_grad_norm = 1
log_interval = 200
learning_rate =  5e-5
num_workers = 0

In [ ]:
import os

TASK = 'D'            # 'D' weathering condition, 'S' strength, 'F' discontinuity spacing
DATA_DIR = 'data'      # Labeled_data_UTF8_{D,S,F}.csv with columns Text, Score (1-5)
MODEL_DIR = 'trained_model'
os.makedirs(os.path.join(MODEL_DIR, TASK), exist_ok=True)
model_save_path = os.path.join(MODEL_DIR, TASK, 'best_model_lr{}_bs{}_epoch{}.pt'.format(learning_rate, batch_size, num_epochs))

In [ ]:
text_data = pd.read_csv(os.path.join(DATA_DIR, 'Labeled_data_UTF8_{}.csv'.format(TASK)))

In [ ]:
text_data.loc[(text_data['Score'] == 1), 'Score'] = 0
text_data.loc[(text_data['Score'] == 2), 'Score'] = 1
text_data.loc[(text_data['Score'] == 3), 'Score'] = 2
text_data.loc[(text_data['Score'] == 4), 'Score'] = 3
text_data.loc[(text_data['Score'] == 5), 'Score'] = 4

data_list = []
for q, label in zip(text_data['Text'], text_data['Score'])  :
    data = []
    data.append(q)
    data.append(str(label))

    data_list.append(data)

In [ ]:
dataset_train, dataset_temp = train_test_split(data_list, test_size=0.3, random_state=0)
dataset_val, dataset_test = train_test_split(dataset_temp, test_size=0.5, random_state=0)
# print(len(dataset_train))
# print(len(dataset_test))

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, dataset, sent_idx, label_idx, bert_tokenizer, max_len,
                 pad, pair):
        transform = nlp.data.BERTSentenceTransform(
            bert_tokenizer, max_seq_length=max_len, pad=pad, pair=pair)

        self.sentences = [transform([i[sent_idx]]) for i in dataset]
        self.labels = [np.int32(i[label_idx]) for i in dataset]

    def __getitem__(self, i):
        return (self.sentences[i] + (self.labels[i], ))

    def __len__(self):
        return (len(self.labels))

In [ ]:
tokenizer = get_tokenizer()
tok = nlp.data.BERTSPTokenizer(tokenizer, vocab, lower=False)

data_train = BERTDataset(dataset_train, 0, 1, tok, max_len, True, False)
data_val = BERTDataset(dataset_val, 0, 1, tok, max_len, True, False)
data_test = BERTDataset(dataset_test, 0, 1, tok, max_len, True, False)

In [ ]:
# data_train[0]

In [ ]:
train_dataloader = torch.utils.data.DataLoader(data_train, batch_size=batch_size, num_workers=num_workers)
val_dataloader = torch.utils.data.DataLoader(data_val, batch_size=batch_size, num_workers=num_workers)
test_dataloader = torch.utils.data.DataLoader(data_test, batch_size=batch_size, num_workers=num_workers)

In [ ]:
class BERTClassifier(nn.Module):
    def __init__(self,
                 bert,
                 hidden_size = 768,
                 num_classes=5,   ##클래스 수 조정##
                 dr_rate=None,
                 params=None):
        super(BERTClassifier, self).__init__()
        self.bert = bert
        self.dr_rate = dr_rate
                 
        self.classifier = nn.Linear(hidden_size , num_classes)
        if dr_rate:
            self.dropout = nn.Dropout(p=dr_rate)
    
    def gen_attention_mask(self, token_ids, valid_length):
        attention_mask = torch.zeros_like(token_ids)
        for i, v in enumerate(valid_length):
            attention_mask[i][:v] = 1
        return attention_mask.float()

    def forward(self, token_ids, valid_length, segment_ids):
        attention_mask = self.gen_attention_mask(token_ids, valid_length)
        
        _, pooler = self.bert(input_ids = token_ids, token_type_ids = segment_ids.long(), attention_mask = attention_mask.float().to(token_ids.device))
        if self.dr_rate:
            out = self.dropout(pooler)
        return self.classifier(out)

In [ ]:
#BERT 모델 불러오기
model = BERTClassifier(bertmodel,  dr_rate=0.5).to(device)

#optimizer와 schedule 설정
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]

optimizer = AdamW(optimizer_grouped_parameters, lr=learning_rate)
loss_fn = nn.CrossEntropyLoss()

t_total = len(train_dataloader) * num_epochs
warmup_step = int(t_total * warmup_ratio)

scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_step, num_training_steps=t_total)

#정확도 측정을 위한 함수 정의
def calc_accuracy(X,Y):
    max_vals, max_indices = torch.max(X, 1)
    train_acc = (max_indices == Y).sum().data.cpu().numpy()/max_indices.size()[0]
    return train_acc
    
train_dataloader

In [ ]:
best_val_acc = 0.0
best_val_epoch = 0


for e in range(num_epochs):
    train_acc = 0.0
    val_acc = 0.0
    model.train()

    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(train_dataloader)):
        optimizer.zero_grad()
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        loss = loss_fn(out, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        optimizer.step()
        scheduler.step()
        train_acc += calc_accuracy(out, label)
#         if batch_id % log_interval == 0:
#             print("epoch {} batch id {} loss {} train acc {}".format(e+1, batch_id+1, loss.data.cpu().numpy(), train_acc / (batch_id+1)))
    print("epoch {} train acc {}".format(e+1, train_acc / (batch_id+1)))
    
    model.eval()

    for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(val_dataloader)):
        token_ids = token_ids.long().to(device)
        segment_ids = segment_ids.long().to(device)
        valid_length= valid_length
        label = label.long().to(device)
        out = model(token_ids, valid_length, segment_ids)
        val_acc += calc_accuracy(out, label)
        
    epoch_val_acc = val_acc / (batch_id+1)
    print("epoch {}: val accuracy {}".format(e+1, epoch_val_acc))
    
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_val_epoch = e
        
#         current_model_save_path = model_save_path.format(learning_rate, batch_size, e+1)
        torch.save(model.state_dict(), model_save_path)  # 모델 상태 저장
        print(f"Best model saved with accuracy: {best_val_acc:.4f} at epoch {best_val_epoch+1}")
    
print(f"Best validation accuracy was {best_val_acc:.4f} at epoch {best_val_epoch+1}")


In [ ]:
# 모델 초기화
model = BERTClassifier(bertmodel, dr_rate=0.5).to(device)

model.load_state_dict(torch.load(model_save_path))

all_preds = []
all_labels = []

model.eval()
test_acc = 0.0
total_count = 0

for batch_id, (token_ids, valid_length, segment_ids, label) in enumerate(tqdm_notebook(test_dataloader)):
    token_ids = token_ids.long().to(device)
    segment_ids = segment_ids.long().to(device)
    valid_length = valid_length
    label = label.long().to(device)
    
    with torch.no_grad():
        out = model(token_ids, valid_length, segment_ids)
    
    preds = torch.max(out, 1)[1].detach().cpu().numpy()  # 예측
    labels = label.detach().cpu().numpy()  # 실제
    
    correct_count = (preds == labels).sum()
    test_acc += correct_count
    total_count += labels.shape[0]
    
    all_preds.extend(preds)
    all_labels.extend(labels)
    
all_labels_list = [0, 1, 2, 3, 4]
conf_matrix = confusion_matrix(all_labels, all_preds, labels=all_labels_list)

final_test_acc = test_acc / total_count
print("final test acc: {}".format(final_test_acc))

In [ ]:
x_axis_labels = ['1', '2', '3', '4', '5']
y_axis_labels = ['1', '2', '3', '4', '5']

plt.figure(figsize=(6,4))
ax = sns.heatmap(conf_matrix, xticklabels=x_axis_labels, yticklabels=y_axis_labels, annot=True, fmt='.0f', annot_kws={"size": 15}, 
                robust=True, cmap='Blues', linewidth=1, square=True)

ax.set_xticklabels(labels=ax.get_xticklabels(), rotation=0, size=15)
ax.set_yticklabels(labels=ax.get_yticklabels(), va='center', rotation=0, size=15)
ax.set_xlabel('Predicted', size=15)
ax.set_ylabel('True', size=15)
plt.title('Confusion Matrix (Count)', size=15)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)

plt.show()



In [ ]:
np.seterr(divide='ignore', invalid='ignore')
confusion_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]
confusion_normalized = np.nan_to_num(confusion_normalized)

In [ ]:
x_axis_labels = ['1', '2', '3', '4', '5']
y_axis_labels = ['1', '2', '3', '4', '5']

plt.figure(figsize=(6,4))
ax = sns.heatmap(confusion_normalized, xticklabels=x_axis_labels, yticklabels=y_axis_labels, annot=True, fmt='.2f', annot_kws={"size": 15}, 
                robust=True, cmap='Blues', linewidth=1, square=True)

ax.set_xticklabels(labels=ax.get_xticklabels(), rotation=0, size=15)
ax.set_yticklabels(labels=ax.get_yticklabels(), va='center', rotation=0, size=15)
ax.set_xlabel('Predicted', size=15)
ax.set_ylabel('True', size=15)
plt.title('Confusion Matrix (Probability)', size=15)

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)

plt.show()